# 04 — Validation Strategy Comparison (Phase 4)

Compares holdout, 5-fold CV, 10-fold CV, and LOOCV on the top 3 models.

**Experiment:** VAL-1

**Source:** `outputs/phase4/val1_summary.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
P4 = ROOT / 'outputs' / 'phase4'
val1 = pd.read_csv(P4 / 'val1_summary.csv')

## VAL-1: Strategy Comparison (v_out_mean)

In [ ]:
vout = val1[val1['output'] == 'v_out_mean']
pivot = vout.pivot_table(index='strategy', columns='model', values='rmse_mean')
pivot_std = vout.pivot_table(index='strategy', columns='model', values='rmse_std')

print('RMSE mean:')
print(pivot.to_string())
print()
print('RMSE std:')
print(pivot_std.to_string())

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
strategies = ['holdout_80_20', '5fold_cv', '10fold_cv', 'loocv']
models = ['GP-M52', 'GP-SE', 'GP-SE-noARD']
colors = ['C0', 'C1', 'C2']
width = 0.2
x = np.arange(len(strategies))

for i, model in enumerate(models):
    means = [vout[(vout['strategy']==s) & (vout['model']==model)]['rmse_mean'].values[0] for s in strategies]
    stds = [vout[(vout['strategy']==s) & (vout['model']==model)]['rmse_std'].values[0] if not pd.isna(vout[(vout['strategy']==s) & (vout['model']==model)]['rmse_std'].values[0]) else 0 for s in strategies]
    ax.bar(x + i*width, means, width, yerr=stds, label=model, color=colors[i], alpha=0.8, capsize=3)

ax.set_xticks(x + width)
ax.set_xticklabels(strategies, rotation=15)
ax.set_ylabel('RMSE (V)')
ax.set_title('VAL-1: Validation Strategy Comparison (v_out_mean)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Conclusions

- All strategies agree: GP-M52 is the best model
- LOOCV gives lowest estimates (~0.1V) but no variance
- Holdout has 3x the variance of 10-fold CV

**Decision:** 10-fold CV (5 repeats) — best balance of bias, variance, and data efficiency.